# Reconhecimento de Gestos com OpenCV e MediaPipe

Este notebook demonstra como realizar o reconhecimento de gestos das mãos em tempo real usando a webcam, OpenCV e MediaPipe.
Utiliza `mp.solutions.hands` para rastreamento simultâneo de duas mãos.

In [39]:
import warnings
warnings.filterwarnings('ignore')

import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print('Imports carregados com sucesso!')

Imports carregados com sucesso!


In [40]:
def classify_gesture(hand_landmarks, handedness):
    """Classifica o gesto da mão baseado nos dedos levantados."""
    tips = [4, 8, 12, 16, 20]
    pip_joints = [3, 6, 10, 14, 18]

    fingers_up = []

    # Polegar - compara coordenada X
    is_right = handedness == 'Right'
    if is_right:
        fingers_up.append(
            hand_landmarks.landmark[tips[0]].x < hand_landmarks.landmark[pip_joints[0]].x
        )
    else:
        fingers_up.append(
            hand_landmarks.landmark[tips[0]].x > hand_landmarks.landmark[pip_joints[0]].x
        )

    # Outros dedos - ponta acima da junta PIP = levantado
    for i in range(1, 5):
        fingers_up.append(
            hand_landmarks.landmark[tips[i]].y < hand_landmarks.landmark[pip_joints[i]].y
        )

    count = sum(fingers_up)

    if count == 0:
        return 'Closed_Fist'
    elif count == 5:
        return 'Open_Palm'
    elif fingers_up == [False, True, False, False, False]:
        return 'Pointing_Up'
    elif fingers_up == [False, True, True, False, False]:
        return 'Victory'
    elif fingers_up == [True, False, False, False, False]:
        if hand_landmarks.landmark[tips[0]].y < hand_landmarks.landmark[pip_joints[0]].y:
            return 'Thumbs_Up'
        else:
            return 'Thumbs_Down'
    elif fingers_up == [True, True, False, False, True]:
        return 'ILoveYou'
    elif fingers_up == [False, True, False, False, True]:
        return 'Rock'
    else:
        return f'{count}_Dedos'

print('Funcao de classificacao definida!')

Funcao de classificacao definida!


In [41]:
# Iniciar a captura da webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print('ERRO: Nao foi possivel acessar a webcam.')
else:
    print(f'Webcam aberta: {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}')
    print('Pressione q na janela de video para encerrar.')

    hands = mp_hands.Hands(
        static_image_mode=False,
        model_complexity=0,
        max_num_hands=2,
        min_detection_confidence=0.3,
        min_tracking_confidence=0.3
    )

    try:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                print('Erro ao ler frame.')
                break

            # Espelhar
            frame = cv2.flip(frame, 1)

            # BGR -> RGB para MediaPipe
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb_frame.flags.writeable = False

            # Processar
            results = hands.process(rgb_frame)

            rgb_frame.flags.writeable = True

            if results.multi_hand_landmarks:

                for idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
                    handedness = results.multi_handedness[idx].classification[0].label

                    # Desenhar landmarks
                    mp_drawing.draw_landmarks(
                        frame,
                        hand_landmarks,
                        mp_hands.HAND_CONNECTIONS,
                        mp_drawing_styles.get_default_hand_landmarks_style(),
                        mp_drawing_styles.get_default_hand_connections_style())

                    # Classificar gesto
                    gesture = classify_gesture(hand_landmarks, handedness)
                    score = results.multi_handedness[idx].classification[0].score
                    text = f'{handedness}: {gesture} ({score:.2f})'

                    # Posicao do texto
                    h, w, c = frame.shape
                    x = int(hand_landmarks.landmark[0].x * w)
                    y = int(hand_landmarks.landmark[0].y * h)

                    # Texto com sombra
                    cv2.putText(frame, text, (max(0, x - 30), max(30, y - 30)),
                                cv2.FONT_HERSHEY_DUPLEX, 0.7, (0, 0, 0), 3, cv2.LINE_AA)
                    cv2.putText(frame, text, (max(0, x - 30), max(30, y - 30)),
                                cv2.FONT_HERSHEY_DUPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)

            cv2.imshow('Reconhecimento de Gestos - 2 Maos', frame)

            if cv2.waitKey(5) & 0xFF == ord('q'):
                break

    finally:
        hands.close()
        cap.release()
        cv2.destroyAllWindows()
        print('Webcam encerrada e janelas fechadas.')

Webcam aberta: 640x480
Pressione q na janela de video para encerrar.
Webcam encerrada e janelas fechadas.
